# High-z Example 15: CG32 and DC396844 — Mdyn < M* Anomalies

**EPS Research — High-z Kinematic Corpus Z1**

Two Z1 galaxies have dynamical mass below stellar mass:
- CG32: log Mdyn - log M* = -0.16 dex
- DC396844: log Mdyn - log M* = -0.35 dex

This is physically implausible. Likely causes:
- Inclination uncertainty (DC396844 has e_inc = 31°)
- 2-ring boundary constraint amplifying errors
- Unresolved non-circular motions

Values reported as published in Jones et al. (2021).

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.20369285  
**arXiv:** 2605.25339  
**Source:** Jones et al. (2021), MNRAS 507, 3540; Le Fevre et al. (2020)  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('high_z_kinematic_corpus_Z1.json') as f:
    corpus = json.load(f)

galaxies  = corpus['galaxies']
rotators  = [g for g in galaxies if g.get('is_rotator') and g.get('quality_tier')==1]
print(f"Total galaxies: {len(galaxies)}")
print(f"Tier-1 rotators: {len(rotators)}")


In [ ]:
import numpy as np
anomalies = []
normal    = []
for g in rotators:
    mdyn  = g.get('log_mdyn_msun')
    mstar = g.get('log_mstar_msun')
    inc   = g.get('inc_kin_deg')
    einc  = g.get('e_inc_kin_deg')
    if mdyn and mstar:
        entry = {'galaxy': g['galaxy'], 'z': g['redshift'],
                 'log_mdyn': mdyn, 'log_mstar': mstar,
                 'diff': mdyn - mstar, 'inc': inc, 'einc': einc}
        if mdyn < mstar:
            anomalies.append(entry)
        else:
            normal.append(entry)

print(f"Anomalous (Mdyn < M*): {len(anomalies)}")
print(f"Normal (Mdyn >= M*):   {len(normal)}")
print()
for a in anomalies:
    print(f"Galaxy: {a['galaxy']}  z={a['z']:.4f}")
    print(f"  log Mdyn = {a['log_mdyn']:.3f}")
    print(f"  log M*   = {a['log_mstar']:.3f}")
    print(f"  Deficit  = {a['diff']:.3f} dex")
    if a['einc']:
        print(f"  Inc uncertainty = ±{a['einc']}° (likely cause)")
    print()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for entry in normal:
    ax.scatter(entry['log_mstar'], entry['log_mdyn'], s=60,
               color='#2166ac', alpha=0.8, zorder=3)
for entry in anomalies:
    ax.scatter(entry['log_mstar'], entry['log_mdyn'], s=120,
               color='#e74c3c', alpha=0.9, zorder=4,
               marker='*', label=f"{entry['galaxy']} (anomalous)")
    ax.annotate(entry['galaxy'], (entry['log_mstar'], entry['log_mdyn']),
                textcoords='offset points', xytext=(5, -10), fontsize=8,
                color='#e74c3c')

all_vals = [e['log_mstar'] for e in normal+anomalies] +            [e['log_mdyn']  for e in normal+anomalies]
lim = [min(all_vals)-0.1, max(all_vals)+0.1]
ax.plot(lim, lim, 'k--', lw=1, alpha=0.4, label='Mdyn = M*')
ax.set_xlabel(r'log M$_*$', fontsize=12); ax.set_ylabel(r'log M$_{m dyn}$', fontsize=12)
ax.set_title('CG32 & DC396844: Mdyn < M* Anomalies\n'
             'Likely inclination uncertainty | Values as published', fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('hz15_mdyn_anomalies.png', dpi=150, bbox_inches='tight')
plt.show()